In [8]:
%pip install opendatasets --upgrade

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
ratings = pd.read_csv('/content/ratings.csv')

In [11]:
user_item = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [12]:
# Similarity in-between users
user_similarity = cosine_similarity(user_item)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item.index,
    columns=user_item.index
)

In [13]:
user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [14]:
def get_similar_users(user_id, user_similarity_df, top_n=5):
    # Select the column for the target user
    # Drop the target user's similarity with themselves (which is 1.0)
    user_similarities = user_similarity_df.loc[user_id]

    # Sort by highest similarity
    top_similar_users = user_similarities.sort_values(ascending=False).drop(user_id).head(top_n)

    return top_similar_users

In [15]:
# Recommendation function to recommend movies based on similar users
def recommend_movies(user_id, user_item, user_similarity_df, top_n_similar=5, top_n_recommendations=5):
    # Get similar users
    similar_users = get_similar_users(user_id, user_similarity_df, top_n=top_n_similar)

    # Get the movies rated by the target user
    user_movies = user_item.loc[user_id]

    # Calculate weighted ratings from similar users
    weighted_ratings = np.zeros(user_item.shape[1])
    for sim_user_id, similarity in similar_users.items():
        sim_user_ratings = user_item.loc[sim_user_id]
        weighted_ratings += similarity * sim_user_ratings.values

    # Normalize by the sum of similarities
    sum_similarities = similar_users.sum()
    if sum_similarities > 0:
        weighted_ratings /= sum_similarities

    # Get movie indices that the target user has not rated
    unrated_movies = user_movies[user_movies == 0].index

    # Recommend movies with the highest weighted ratings among unrated movies
    recommendations = pd.Series(weighted_ratings, index=user_item.columns)
    recommendations = recommendations[unrated_movies].sort_values(ascending=False).head(top_n_recommendations)

    return recommendations

In [16]:
user_id = 10
recommend_movies(user_id, user_item, user_similarity_df)

,0
movieId,
78499,2.889794
69757,2.816518
72641,2.671646
318,2.648368
1721,2.558216
